---
title: Estimación de series temporales de rasgos biofísicos
subject: Ejercicio
subtitle: Ejercicio que muestra cómo obtener imágenes mensuales de rasgos biofísicos mediante imágenes Sentinel-2
authors:
  - name: Héctor Nieto
    affiliations:
      - Instituto de Ciencias Agrarias, ICA
      - CSIC
    orcid: 0000-0003-4250-6424
    email: hector.nieto@ica.csic.es
  - name: Radoslaw Guzinski
    affiliations:
      - DHI
    orcid: 0000-0003-0044-6806
  - name: Benjamin Mary
    affiliation:
      - Instituto de Ciencias Agrarias
      - CSIC
    orcid: 0000-0003-0815-842X
label: nb-biophysical
license: CC-BY-SA-4.0
keywords: Prospect, 4SAIL, crop yield, Daisy crop model
myst:
  enable_extensions: ["deflist", "attrs_block", "attrs_inline"]
jupytext:
  text_representation:
    extension: .md
    format_name: myst
    format_version: 0.13
    jupytext_version: 1.19.1
kernelspec:
  display_name: Python 3 (ipykernel)
  language: python
  name: python3
---

# Introducción

En este ejercicios vamos a pre-procesar en la nube y descargar imágenes mensuales para los biofísicos obtenidos a partir de imágenes Sentinel-2 con el fin de evaluar la evolución temporal de un conjunto de rodales/áreas de interés, y así poder detectar anomalías y tendencias tras una perturbación y/o tratamiento..

Usaremos para ello el entortno del [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) usando la interfaz [openEO](https://openeo.org/).

Este cuaderno puede ejecutarse en el [Jupyterhub de Copernicus Dataspace](https://jupyterhub.dataspace.copernicus.eu), en cuyo caso no se realizan descargas locales de datos, ya que tanto los datos como el entorno de ejecución están en CDSE y se mantienen en tu cuenta.

:::{warning} Atención
Si estás usando el entorno de CDSE debes seleccionar uno de los kernels con GDAL instalado, p. ej. "Geo science".
:::

::::{note} Nota
Las características de Sentinel-2 son las siguientes
:::{table} Características de la misión Sentinel-2
:label: s2
Plataformas | Rango espectral     | Número de bandas | Resolución espacial | Resolución temporal
:---        | :---                | :---             | :---                | :---            
A, B, C     | Visible, NIR, SWIR  | 10 (13)          | 10 -- 20 m          | 5 -- 10 días
:::
::::

Primero comprobamos que el Sen-ET Toolbox esté instalado (y lo instalamos si es necesario) y luego importamos todos los paquetes necesarios.

In [ ]:
try:
    import senet_toolbox
    print("senet_toolbox importado correctamente")
except ModuleNotFoundError:
    print("Falta la librería senet_toolbox, instalando desde Git")
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

In [ ]:
from pathlib import Path
from dateutil.relativedelta import relativedelta
from shapely import to_geojson
from shapely.geometry import box
from osgeo import gdal
import rasterio
import openeo
import numpy as np
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
from joblib import load
from rasterstats import zonal_stats
from senet_toolbox.workflows import collect_input_data
from senet_toolbox.utils import visualization, date_selector
from senet_toolbox.utils.raster_utils import save_raster
from senet_toolbox.workflows import biophysical_processing
from ipywidgets import interact, interactive, fixed, widgets
from IPython.display import display
from statsmodels.tsa.seasonal import MSTL
import datetime as dt
print("Librerías importadas correctamente, puedes continuar")

### Seleccionar el Área de Interés
Para mantener los datos organizados y facilitar el procesamiento de series temporales, los datos de entrada y salida se guardan en carpetas de Área de Interés (AOI). Todos los datos dentro de una carpeta AOI tienen la misma extensión y cuadrícula.

En la celda siguiente, selecciona la ubicación donde deseas almacenar los datos y el nombre del AOI. Al ejecutar en el Jupyterhub de CDSE, se recomienda mantenerlo dentro de `./mystorage/301-biophysical`, de lo contrario los datos se borrarán entre sesiones.

Si estás configurando un nuevo AOI, dibuja un polígono en el mapa con la extensión que deseas procesar. Se recomienda seleccionar AOIs de pequeñas (unos pocos kilómetros) para agilizar el procesado y no usar los cŕeditos gratuitos rápidamente

Si estás trabajando con un AOI existente, el mapa mostrará su extensión.

In [ ]:
data_dir = "./mystorage/301a-biophysical"
aoi_name = "agramon"
aoi_data_dir = Path(data_dir) / aoi_name
print(f"Usando carpeta de trabajo en {aoi_data_dir}")

In [ ]:
# Dibuja o visualiza la extensión del AOI al configurar uno nuevo
map, bboxs = visualization.select_aoi(aoi_data_dir)
map

## Seleccionar el rango de fechas
En la siguiente celda selecciona el año hidrológico de inicio y de final que te interese procesar

In [ ]:
w_years = widgets.IntRangeSlider(
    description="Años",
    tooltip='Selecciona el rango de años hidrológicos a procesar',
    disabled=False,
    min=2015,
    max=dt.datetime.today().year - 1,
    value=(2015, dt.datetime.today().year - 1),
)
display(w_years)

## Conectarse al backend de OpenEO

Las imágenes de Sentinel-2 serán procesadas y descargadas desde la interfaz OpenEO de CDSE. Ejecuta la celda siguiente para autenticarte en OpenEO.

:::{important} Importante
Es posible que debas hacer clic en un enlace de autenticación que aparecerá y seguir las instrucciones.
:::

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

## Descargar series temporales de Sentinel-2

Descargaremos medias mensuales y máximos anuales del LAI y del Contenido de Clorofila del Dosel, que se obtienen a partir del producto de reflectancia en el Fondo de la Atmósfera (BOA) de Sentinel-2 usando el [procesador BIOPAR de OpenEO](https://openeo.dataspace.copernicus.eu/openeo/1.1/processes/u:3e24e251-2e9a-438f-90a9-d4500e576574/BIOPAR):

Tanto el LAI como el CCC se definen en BIOPAR de la siguiente manera:
- **Índice de Área Foliar (LAI)**: la mitad del área total de los elementos verdes del dosel por unidad de superficie horizontal del suelo. El valor derivado por satélite corresponde al LAI verde total de todas las capas del dosel, incluido el sotobosque, que puede representar una contribución muy significativa, especialmente en bosques.

- **Contenido de Clorofila del Dosel (CCC)**: el contenido total de clorofila por unidad de superficie del suelo en un grupo continuo de plantas. Es muy adecuado para cuantificar el contenido de nitrógeno a nivel del dosel y estimar la producción primaria bruta.

La metodología BIOPAR fue desarrollada inicialmente para generar productos biofísicos a partir de los sensores SPOT-VEGETATION, ENVISAT-MERIS, SPOT-HRVIR y LANDSAT-OLI, y fue posteriormente adaptada para Sentinel-2. Consiste principalmente en simular una base de datos exhaustiva de reflectancias de dosel (BOA) a partir de las características de la vegetación y la geometría de observación e iluminación. A continuación, se entrenan redes neuronales para estimar una serie de estas características del dosel (BIOPARs) a partir de las reflectancias BOA simuladas junto con los ángulos que definen la configuración observacional.

:::{seealso} Ver también
[Weiss y Baret (2016). S2ToolBox Level 2 products: LAI, FAPAR, FCOVER Version 1.1](http://step.esa.int/docs/extra/ATBD_S2ToolBox_L2B_V1.1.pdf)
:::

:::{important} Importante
Para áreas grandes, la descarga y agregación de datos en OpenEO puede tardar bastante y podría fallar. Se recomienda procesar regiones más pequeñas a la vez.
Accede a [https://openeo.dataspace.copernicus.eu/](https://openeo.dataspace.copernicus.eu/) e inicia sesión para hacer seguimiento de los trabajos y ver posibles errores.
:::

### Descargar promedios mensuales de LAI

In [ ]:
MAX_JOBS = 24
        
date_ini = dt.datetime(w_years.value[0], 10, 1)
date_end = dt.datetime(w_years.value[1], 9, 30)
bbox = bboxs[0]
bbox_polygon = eval(to_geojson(box(*bbox)))
var = "LAI"
input_dir = aoi_data_dir / "input"

if not input_dir.exists():
    input_dir.mkdir()

time_window = [str(date_ini.date()), str(date_end.date())]
print(f"Procesando BIOPAR {var} desde {time_window[0]} hasta {time_window[1]}\n"
      f"para la extensión {bbox}, esto puede tardar un momento")

bio = biophysical_processing.get_biopar(
    connection, var, time_window, bbox_polygon
    )

date = date_ini
jobs = []
count = 0
while date < date_end:    
    dates_range = [str(date.date()),  
                   str((date + relativedelta(months=1) - dt.timedelta(days=1)).date())]    
    bio_month = bio.filter_temporal(dates_range).reduce_dimension(dimension="t", reducer="mean")  

    s2_path = input_dir / f"s2_{date:%Y%m}_{var}.tif"
    if not s2_path.exists():
        print(f"Creando trabajo para el {var} mensual de Sentinel-2 desde {dates_range[0]} hasta {dates_range[1]}")
        job = bio_month.create_job(out_format="GTiff")
        job.start()
        jobs.append([job, s2_path])
        count += 1
    else:
        print(f"Se encontraron datos de Sentinel-2 en caché para el mes {date:%m} de {date:%Y}. Omitiendo descarga.")  
    
    date += relativedelta(months=1)
    count += 1
    # Download data every MAX_DOWNLOADS
    if count > MAX_JOBS:
        for job, path in jobs:
            print(f"Procesando y descargando en {path}", end="...") 
            collect_input_data.wait_and_download(job, path, poll_interval=60)
            print(f"Descargado") 
        # Restart job list and counter
        count = 0        
        jobs = []

# Process the last batch of jobs
for job, path in jobs:
    print(f"Procesando y descargando en {path}", end="...") 
    collect_input_data.wait_and_download(job, path, poll_interval=60)
    print(f"Descargado") 


print(f"Todos las imágenes {var} procesadas, puedes continuar")

:::{attention} Atención
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta la celda [](#remove-jobs) para eliminar todos los trabajos pendientes de la nube y reintentar
:::

### Descargar promedios mensuales de Contenido de Clorofila en el dosel
Haz lo mismo para el producto CCC

In [ ]:
MAX_JOBS = 24
        
date_ini = dt.datetime(w_years.value[0], 10, 1)
date_end = dt.datetime(w_years.value[1], 9, 30)
bbox = bboxs[0]
bbox_polygon = eval(to_geojson(box(*bbox)))
var = "CCC"
input_dir = aoi_data_dir / "input"

if not input_dir.exists():
    input_dir.mkdir()

time_window = [str(date_ini.date()), str(date_end.date())]
print(f"Procesando BIOPAR {var} desde {time_window[0]} hasta {time_window[1]}\n"
      f"para la extensión {bbox}, esto puede tardar un momento")

bio = biophysical_processing.get_biopar(
    connection, var, time_window, bbox_polygon
    )

date = date_ini
jobs = []
count = 0
while date < date_end:    
    dates_range = [str(date.date()),  
                   str((date + relativedelta(months=1) - dt.timedelta(days=1)).date())]    
    bio_month = bio.filter_temporal(dates_range).reduce_dimension(dimension="t", reducer="mean")  

    s2_path = input_dir / f"s2_{date:%Y%m}_{var}.tif"
    if not s2_path.exists():
        print(f"Creando trabajo para el {var} mensual de Sentinel-2 desde {dates_range[0]} hasta {dates_range[1]}")
        job = bio_month.create_job(out_format="GTiff")
        job.start()
        jobs.append([job, s2_path])
        count += 1
    else:
        print(f"Se encontraron datos de Sentinel-2 en caché para el mes {date:%m} de {date:%Y}. Omitiendo descarga.")         
    
    date += relativedelta(months=1)
    
    # Download data every MAX_DOWNLOADS
    if count > MAX_JOBS:
        for job, path in jobs:
            print(f"Procesando y descargando en {path}", end="...") 
            collect_input_data.wait_and_download(job, path, poll_interval=60)
            print(f"Descargado") 
        # Restart job list and counter
        count = 0        
        jobs = []

# Process the last batch of jobs
for job, path in jobs:
    print(f"Procesando y descargando en {path}", end="...") 
    collect_input_data.wait_and_download(job, path, poll_interval=60)
    print(f"Descargado") 


print(f"Todos las imágenes {var} procesadas, puedes continuar")

:::{attention} Atención
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta la celda [](#remove-jobs) para eliminar todos los trabajos pendientes de la nube y reintentar
:::

## De CCC a clorofila foliar
Del contenido de clorofila en el dosel `CCC` y el Índice de área foliar `LAI` podemos derivar la clorifila foliar:

:::{math}
C_{a+b} \left(\mu\textrm{g m}^{-2}\right)= \frac{CCC \left(\mu\textrm{g m}^{-2}\right)}{LAI}
:::


In [ ]:
def read_raster(path):
    src = rasterio.open(path)
    profile = src.meta
    values = src.read(1)
    src.close()
    return values, profile 

lai_images = sorted(list(input_dir.glob(f"s2_*_LAI.tif")))
for lai_path in lai_images:
    date = lai_path.stem.split("_")[1]
    ccc_path = input_dir / f"s2_{date}_CCC.tif"
    cab_path = input_dir / f"s2_{date}_CAB.tif"
    if cab_path.exists():
        print(f"{cab_path} ya generado, omitiendo")
    else:
        lai, profile = read_raster(lai_path)
        ccc, _ = read_raster(ccc_path)
        cab = ccc / lai
        save_raster(cab_path, cab, profile)

print("Todos los productos de clorifila foliar calculados")

# Extraer tendencias y estacionalidad

## Selecciona una capa
Por defecto, las tendencias se van a calcular para el promedio de todos los píxeles (según la extensión que dibujaste anteriormente)

En esta celda puedes en cambio subir una capa geojson con los polígonos sobre los que quieres hacer los cálculos. De este modo se sacarían las tendencias promedio

In [ ]:
w_file = widgets.FileUpload(
    value = (),
    accept='.geojson',
    multiple=False
)
display(w_file)

### Selecciona variable a procesar

In [ ]:
w_var = widgets.Dropdown(
    options=["LAI", "CAB", "CCC"],
    value='LAI',
    description='Variable:',
    tooltip="Selecciona rasgo biofísico a procesar")
display(w_var)

In [ ]:
var = w_var.value
if len(w_file.value) == 0:
    print("Archivo de capas no aportado, se usará el promedio de toda la escena")
else:
    uploaded_file = w_file.value[0]    
    ext = uploaded_file.name.split(".")[-1]
    geo_file = aoi_data_dir / f"{aoi_name}.{ext}"
    with open(geo_file, "wb") as fp:
        fp.write(uploaded_file.content.tobytes())

    site_data = gpd.read_file(geo_file).explode()

dates = []
df = {"date":[], "fid": [], "value": []}
images = sorted(list(input_dir.glob(f"s2_*_{var}.tif")))
if len(w_file.value) == 0:
    for image in images:
        print(f"Calculando el promedio de {image}")
        date = dt.datetime.strptime(image.stem.split("_")[1], "%Y%m")
        src = rasterio.open(image)
        data = np.nanmean(src.read())
        src.close()
        df["date"].append(date)
        df["value"].append(data)
        df["fid"].append(0)

else:
    for image in images:
        print(f"Calculando el promedio zonal de {image}")
        crs = rasterio.open(image).crs
        stats = zonal_stats(site_data.to_crs(crs), 
                            image, stats="mean", geojson_out=True, all_touched=True)
        date = dt.datetime.strptime(image.stem.split("_")[1], "%Y%m")        
        for stat in stats: 
            df["date"].append(date)
            df["value"].append(stat['properties']["mean"])
            df["fid"].append(stat['properties']["fid"])
            
df = pd.DataFrame(df)
fig = go.Figure()
for fid in df["fid"].unique():
    id = df["fid"] == fid
    subset = df.loc[id]
    fig.add_trace(go.Scatter(x=subset["date"], y=subset["value"], name=f"site-{fid}", mode="lines"))
    
fig.update_layout(title_text=f"{var} mensual", xaxis_title="Fecha", yaxis_title=var)

## Descomposición estacional de las extracciones

In [ ]:
stl_kwargs = {"seasonal_deg": 0,
              "trend_deg": 0}

out_dir = aoi_data_dir / "output"
if not out_dir.is_dir():
    out_dir.mkdir(parents=True)
    
out_file = out_dir / f"zonal_trends_{var}.csv"
ts_dict = {"fid": [], "date": [], "values": [], "trend": []}

fig = go.Figure()
for fid in df["fid"].unique():
    id = df["fid"] == fid
    subset = df.loc[id]
    subset = subset.set_index("date")
    subset = subset["value"].interpolate(method='time').bfill()
    ts = MSTL(subset,
              periods=12, windows=5*12+1, iterate=5,
              stl_kwargs=stl_kwargs).fit()
    
    ts_dict["date"] += ts.trend.index.tolist()
    ts_dict["values"] += ts.observed.values.tolist()
    ts_dict["trend"] += ts.trend.values.tolist()
    ts_dict["fid"] += np.full_like(ts.observed.values, fid).tolist()
    fig.add_trace(go.Scatter(x=ts.trend.index, y=ts.trend.values, name=f"site-{fid}", mode="lines"))

ts_dict = pd.DataFrame(ts_dict)
ts_dict.to_csv(out_file, sep=";")
print(f"Guardadas las tendencias en {out_file}")
fig.update_layout(title_text=f"Tendencia anual para {var}", xaxis_title="Fecha", yaxis_title=var)

# Ejercicio
1. Genera tu propia capa con los polígonos/rodales que quieras evaluar.
2. Extrae las series temporales de reflectividad y genera los rasgos biofísicos.
3. Evalúa las series temporales mediante la descomposición estacional
4. Genera un pequeño informe de 1-2 págines detallando conclusiones sobre las tendencias observadas
5. Exporta todo el cuaderno en formato pdf

## Entregables
* Informe de las series temporales y las tendencias obsevadas
* El cuaderno digital en formato pdf

(remove-jobs)=
# Eliminar trabajos actuales
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta esta celda para eliminar todos los trabajos pendientes de la nube y reintentar

In [ ]:
jobs = connection.list_jobs()
for job in jobs:
    if job["status"] == "finished":
        continue
    
    print(f"Borrando trabajo {job}")
    job = connection.job(job["id"])    
    job.delete()

print("Todos los trabajos pendientes en cola eliminados, puedes volver a procesar los productos")